In [ ]:
# =========================
# 0. INSTALLS (Colab only)
# =========================
!pip install pymupdf groq pandas openpyxl

# =========================
# 1. IMPORTS
# =========================
import fitz  # PyMuPDF
import re
import json
import pandas as pd
from groq import Groq

# =========================
# 2. CONFIG
# =========================
API_KEY = "API KEY"  # 🔴 replace this
MODEL = "llama-3.3-70b-versatile"
client = Groq(api_key=API_KEY)

# =========================
# 3. PROMPTS
# =========================
EXTRACT_PROMPT = """
Extract structured data from this technical specification text.

Return STRICT JSON ONLY.

Format:
{{
  "rows": [
    {{
      "Component": "...",
      "Specification": "..."
    }}
  ]
}}

Text:
{chunk}
"""

CLASSIFY_PROMPT = """
Classify each item as Hardware or Software.

Rules:
- Hardware = physical components (CPU, RAM, HDD, RAID, etc.)
- Software = OS, protocols, services, licenses

Return STRICT JSON:
[
  {{
    "Component": "...",
    "Specification": "...",
    "Category": "Hardware or Software"
  }}
]

Data:
{data}
"""

# =========================
# 4. TEXT EXTRACTION
# =========================
def extract_text_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# =========================
# 5. CLEANING
# =========================
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# =========================
# 6. STRUCTURE
# =========================
def structure_lines(text):
    return text.split("  ")  # simple split

# =========================
# 7. CHUNKING
# =========================
def chunk_text(lines, size=2000):
    chunks = []
    current = ""

    for line in lines:
        if len(current) + len(line) < size:
            current += line + "\n"
        else:
            chunks.append(current)
            current = line

    if current:
        chunks.append(current)

    return chunks

# =========================
# 8. EXTRACTION (FIXED)
# =========================
def process_chunk(chunk):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": EXTRACT_PROMPT.format(chunk=chunk)}],
        max_tokens=2048,
    )

    raw = response.choices[0].message.content

    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        data = json.loads(raw)

        if isinstance(data, dict) and "rows" in data:
            return data["rows"]

        elif isinstance(data, list):
            return data

        else:
            print("⚠️ Unexpected format:", raw[:200])
            return []

    except Exception:
        print("❌ Extraction failed")
        print(raw[:300])
        return []

# =========================
# 9. BATCH PROCESS
# =========================
def process_all(chunks):
    all_rows = []

    for chunk in chunks:
        rows = process_chunk(chunk)
        all_rows.extend(rows)

    return all_rows

# =========================
# 10. CLASSIFICATION (NO HARDCODE)
# =========================
def classify_batch(rows):
    if not rows:
        return []

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": CLASSIFY_PROMPT.format(data=json.dumps(rows[:50]))
        }],
        max_tokens=2048,
    )

    raw = response.choices[0].message.content
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        return json.loads(raw)
    except:
        print("❌ Classification failed")
        print(raw[:300])
        return rows

# =========================
# 11. EXPORT
# =========================
def export(rows, path="output.xlsx"):
    df = pd.DataFrame(rows)

    if df.empty:
        print("❌ No data to export")
        return

    for col in ["Component", "Specification", "Category"]:
        if col not in df.columns:
            df[col] = ""

    with pd.ExcelWriter(path) as writer:
        df[df["Category"] == "Hardware"].to_excel(writer, "Hardware", index=False)
        df[df["Category"] == "Software"].to_excel(writer, "Software", index=False)

    print("✅ Saved:", path)

# =========================
# 12. RUN PIPELINE
# =========================
def run(file_path):
    print("[1] Extracting...")
    raw = extract_text_pdf(file_path)

    print("[2] Cleaning...")
    clean = clean_text(raw)

    print("[3] Structuring...")
    structured = structure_lines(clean)

    print("[4] Chunking...")
    chunks = chunk_text(structured)

    print("[5] Extracting...")
    rows = process_all(chunks)

    print("👉 Extracted rows:", len(rows))
    print(rows[:3])

    print("[6] Classifying...")
    rows = classify_batch(rows)

    print("[7] Exporting...")
    export(rows)

    print("✅ Done.")

In [ ]:
from google.colab import files

uploaded = files.upload()
file_path = list(uploaded.keys())[0]

run(file_path)